# 03 A-CDM Event Log Build

This notebook converts turnaround cases into proxy A-CDM milestones, checks timestamp ordering, writes validation outputs, and exports PM4Py-friendly CSV files.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.build_acdm_event_log import build_acdm_event_log, compute_milestones
from src.config import get_data_paths, load_config
from src.export_outputs import export_outputs
from src.utils import read_table
from src.validate_event_log import validate_event_log


In [ ]:
config_path = PROJECT_ROOT / 'configs' / 'default_config.yaml'
config = load_config(config_path)
paths = get_data_paths(config, PROJECT_ROOT)
turnaround_df = read_table(paths['processed_dir'] / 'turnaround_cases.parquet')
turnaround_df.head()


In [ ]:
acdm_cfg = config['processing']['acdm']
preview_log = compute_milestones(
    turnaround_df,
    taxi_in_minutes=int(acdm_cfg['taxi_in_minutes']),
    taxi_out_minutes=int(acdm_cfg['taxi_out_minutes']),
    include_ground_handling_markers=bool(acdm_cfg.get('include_ground_handling_markers', False)),
)
preview_log.head(12)


In [ ]:
event_log = build_acdm_event_log(config_path)
event_counts = event_log.groupby('case_id').size().describe()
ordering_ok = event_log.sort_values(['case_id', 'event_order', 'timestamp']).groupby('case_id')['timestamp'].apply(lambda s: s.is_monotonic_increasing)
print(event_counts)
print(f'Cases with valid ordering: {int(ordering_ok.sum())} / {len(ordering_ok)}')
event_log.head(12)


In [ ]:
validate_event_log(config_path)
export_outputs(config_path)
quality_report = paths['processed_dir'] / 'data_quality_report.md'
print(quality_report)


In [ ]:
pm4py_ready = read_table(paths['processed_dir'] / 'acdm_event_log_xes_ready.csv')
pm4py_ready[['case_id', 'activity', 'timestamp', 'event_order']].head(12)
